<a href="https://colab.research.google.com/github/stephenwacha743-maker/GROUP-ONE-WORK/blob/main/TRAVERSING.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd

def run_colab_traverse():
    # 1. Create sample raw survey data directly in memory
    traverse_data = {
        'Line': ['A-B', 'B-C', 'C-D', 'D-E', 'E-A'],
        'Distance': [125.40, 180.25, 145.80, 210.15, 132.60],
        'Azimuth_Deg': [42.5000, 118.2500, 205.7500, 282.1667, 355.3333]
    }
    df = pd.DataFrame(traverse_data)

    # Starting coordinates for Benchmark Station A
    start_x = 1000.0
    start_y = 1000.0

    # 2. Convert Azimuth to Radians and compute raw Latitudes & Departures
    azimuth_rad = np.radians(df['Azimuth_Deg'])
    distances = df['Distance']

    df['Raw_Lat'] = distances * np.cos(azimuth_rad)
    df['Raw_Dep'] = distances * np.sin(azimuth_rad)

    # 3. Misclosure Analysis
    total_perimeter = distances.sum()
    sum_lat = df['Raw_Lat'].sum()
    sum_dep = df['Raw_Dep'].sum()
    linear_misclosure = np.sqrt(sum_lat**2 + sum_dep**2)
    relative_precision = total_perimeter / linear_misclosure if linear_misclosure != 0 else float('inf')

    print("=" * 65)
    print("           TRAVERSE SUMMARY & MISCLOSURE ANALYSIS          ")
    print("=" * 65)
    print(f"Total Perimeter          : {total_perimeter:.3f} m")
    print(f"Error in Latitude        : {sum_lat:+.4f} m")
    print(f"Error in Departure       : {sum_dep:+.4f} m")
    print(f"Linear Misclosure        : {linear_misclosure:.4f} m")
    print(f"Relative Precision       : 1 : {relative_precision:,.0f}")
    print("=" * 65)

    # 4. Apply Bowditch Adjustments (Compass Rule)
    df['Corr_Lat'] = -sum_lat * (distances / total_perimeter)
    df['Corr_Dep'] = -sum_dep * (distances / total_perimeter)
    df['Adj_Lat'] = df['Raw_Lat'] + df['Corr_Lat']
    df['Adj_Dep'] = df['Raw_Dep'] + df['Corr_Dep']

    # 5. Compute Final Adjusted Station Coordinates
    eastings = [start_x]
    northings = [start_y]
    curr_x, curr_y = start_x, start_y

    for dep, lat in zip(df['Adj_Dep'], df['Adj_Lat']):
        curr_x += dep
        curr_y += lat
        eastings.append(curr_x)
        northings.append(curr_y)

    df['Easting_X'] = eastings[:-1]
    df['Northing_Y'] = northings[:-1]

    results = df[['Line', 'Distance', 'Azimuth_Deg', 'Adj_Lat', 'Adj_Dep', 'Easting_X', 'Northing_Y']]

    print("\nADJUSTED TRAVERSE RESULTS:")
    print(results.round(3).to_string(index=False))

    # 6. Optional: Export to CSV in Colab environment
    results.round(3).to_csv("adjusted_traverse_output.csv", index=False)
    print(f"\n[Success] Results exported to 'adjusted_traverse_output.csv'")

# Execute the function
run_colab_traverse()

           TRAVERSE SUMMARY & MISCLOSURE ANALYSIS          
Total Perimeter          : 794.200 m
Error in Latitude        : +52.2678 m
Error in Departure       : -36.0605 m
Linear Misclosure        : 63.5003 m
Relative Precision       : 1 : 13

ADJUSTED TRAVERSE RESULTS:
Line  Distance  Azimuth_Deg  Adj_Lat  Adj_Dep  Easting_X  Northing_Y
 A-B    125.40       42.500   84.202   90.413   1000.000    1000.000
 B-C    180.25      118.250  -97.178  166.965   1090.413    1084.202
 C-D    145.80      205.750 -140.917  -56.722   1257.378     987.023
 D-E    210.15      282.167   30.460 -195.888   1200.655     846.106
 E-A    132.60      355.333  123.434   -4.768   1004.768     876.566

[Success] Results exported to 'adjusted_traverse_output.csv'


In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output, Markdown
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import files
import io

# --- UI CONTROLS ---
num_stations_widget = widgets.IntText(value=5, description='Number of Legs:', style={'description_width': 'initial'}, layout=widgets.Layout(width='200px'))
start_x_widget = widgets.FloatText(value=1000.0, description='Start Easting (X):', style={'description_width': 'initial'})
start_y_widget = widgets.FloatText(value=1000.0, description='Start Northing (Y):', style={'description_width': 'initial'})
start_azim_widget = widgets.FloatText(value=42.5000, description='Start Azimuth (°):', style={'description_width': 'initial'})

method_widget = widgets.Dropdown(
    options=['Bowditch (Compass Rule)', 'Transit Rule'],
    value='Bowditch (Compass Rule)',
    description='Adjustment Method:',
    style={'description_width': 'initial'}
)

spec_widget = widgets.Dropdown(
    options=[('Cadastral Standard (1:10,000)', 10000), ('Engineering Control (1:20,000)', 20000), ('High-Precision Geodetic (1:50,000)', 50000)],
    value=10000,
    description='Target Spec:',
    style={'description_width': 'initial'}
)

# --- FILE UPLOAD WIDGET ---
upload_widget = widgets.FileUpload(
    accept='.csv',
    multiple=False,
    description='Upload CSV',
    button_style='info',
    tooltip='Upload a CSV with columns: Line, Distance, Angle'
)

form_container = widgets.VBox()
output_area = widgets.Output()

dist_widgets = []
angle_widgets = []

def update_form(change=None):
    global dist_widgets, angle_widgets
    n = num_stations_widget.value
    if n < 3:
        n = 3
        num_stations_widget.value = 3

    dist_widgets = []
    angle_widgets = []
    station_letters = [chr(65 + i) for i in range(n + 1)]

    rows = []
    for i in range(n):
        from_stn = station_letters[i]
        to_stn = station_letters[i+1]
        line_name = f"{from_stn}-{to_stn}"

        d_w = widgets.FloatText(value=125.4 if i==0 else 150.0, description=f'{line_name} Dist:', style={'description_width': 'initial'}, layout=widgets.Layout(width='220px'))
        a_w = widgets.FloatText(value=108.0, description=f'Stn {from_stn} Angle (°):', style={'description_width': 'initial'}, layout=widgets.Layout(width='220px'))

        dist_widgets.append(d_w)
        angle_widgets.append(a_w)
        rows.append(widgets.HBox([d_w, a_w]))

    form_container.children = rows

def handle_upload(change):
    if not upload_widget.value:
        return

    uploaded_file = list(upload_widget.value.values())[0]
    content = uploaded_file['content']
    try:
        df_uploaded = pd.read_csv(io.BytesIO(content))
        # Basic cleaning
        df_uploaded.columns = [c.strip().lower() for c in df_uploaded.columns]

        # Map file columns to UI inputs
        req_cols = ['distance', 'angle']
        if not all(col in df_uploaded.columns for col in req_cols):
            with output_area:
                print("Error: CSV must contain 'Distance' and 'Angle' columns.")
                return

        num_stations_widget.value = len(df_uploaded)
        update_form()

        for i, row in df_uploaded.iterrows():
            dist_widgets[i].value = float(row['distance'])
            angle_widgets[i].value = float(row['angle'])

        with output_area:
            clear_output()
            print(f"Successfully loaded {len(df_uploaded)} stations from '{uploaded_file['metadata']['name']}'!")
    except Exception as e:
        with output_area:
            print(f"Error reading file: {e}")

upload_widget.observe(handle_upload, names='value')
num_stations_widget.observe(update_form, names='value')
update_form()

compute_btn = widgets.Button(description='Generate Report, Plot & Compute', button_style='success', icon='file-alt')

def on_compute_clicked(b):
    with output_area:
        clear_output()
        n = num_stations_widget.value
        station_letters = [chr(65 + i) for i in range(n + 1)]
        lines = [f"{station_letters[i]}-{station_letters[i+1]}" for i in range(n)]

        distances = np.array([w.value for w in dist_widgets])
        measured_angles = np.array([w.value for w in angle_widgets])
        start_x = start_x_widget.value
        start_y = start_y_widget.value
        start_azim = start_azim_widget.value
        selected_method = method_widget.value
        min_spec = spec_widget.value

        # --- 1. ANGULAR DISCREPANCY & BALANCING ---
        theoretical_sum = (n - 2) * 180.0
        sum_measured_angles = measured_angles.sum()
        angular_error_deg = sum_measured_angles - theoretical_sum
        angular_error_sec = angular_error_deg * 3600.0
        correction_per_angle = -angular_error_deg / n
        adjusted_angles = measured_angles + correction_per_angle

        # --- 2. AZIMUTH PROPAGATION ---
        azimuths = [start_azim]
        curr_azim = start_azim
        for i in range(n - 1):
            curr_azim = (curr_azim + 180.0 - adjusted_angles[i]) % 360.0
            azimuths.append(curr_azim)

        df = pd.DataFrame({
            'Line': lines,
            'Distance': distances,
            'Meas_Angle': measured_angles,
            'Adj_Angle': adjusted_angles,
            'Azimuth_Deg': azimuths
        })

        # --- 3. RAW LATITUDES & DEPARTURES ---
        azimuth_rad = np.radians(df['Azimuth_Deg'])
        df['Raw_Lat'] = distances * np.cos(azimuth_rad)
        df['Raw_Dep'] = distances * np.sin(azimuth_rad)

        total_perimeter = distances.sum()
        sum_lat = df['Raw_Lat'].sum()
        sum_dep = df['Raw_Dep'].sum()
        linear_misclosure = np.sqrt(sum_lat**2 + sum_dep**2)
        relative_precision = total_perimeter / linear_misclosure if linear_misclosure != 0 else float('inf')

        # --- 4. ERROR ADJUSTMENT (BOWDITCH vs TRANSIT) ---
        if selected_method == 'Bowditch (Compass Rule)':
            df['Corr_Lat'] = -sum_lat * (distances / total_perimeter)
            df['Corr_Dep'] = -sum_dep * (distances / total_perimeter)
        else:
            sum_abs_lat = np.abs(df['Raw_Lat']).sum()
            sum_abs_dep = np.abs(df['Raw_Dep']).sum()
            df['Corr_Lat'] = -sum_lat * (np.abs(df['Raw_Lat']) / sum_abs_lat) if sum_abs_lat != 0 else 0
            df['Corr_Dep'] = -sum_dep * (np.abs(df['Raw_Dep']) / sum_abs_dep) if sum_abs_dep != 0 else 0

        df['Adj_Lat'] = df['Raw_Lat'] + df['Corr_Lat']
        df['Adj_Dep'] = df['Raw_Dep'] + df['Corr_Dep']

        # --- 5. COORDINATE COMPUTATION ---
        eastings = [start_x]
        northings = [start_y]
        curr_x, curr_y = start_x, start_y
        for dep, lat in zip(df['Adj_Dep'], df['Adj_Lat']):
            curr_x += dep
            curr_y += lat
            eastings.append(curr_x)
            northings.append(curr_y)

        df['Easting_X'] = eastings[:-1]
        df['Northing_Y'] = northings[:-1]

        # --- 6. QC CHECK ---
        qc_status = "PASS" if relative_precision >= min_spec else "FAIL (Exceeds Tolerance)"

        # --- COMPOSE COMPREHENSIVE TEXT REPORT ---
        report_text = f"""=====================================================================
          AUTOMATED CLOSED TRAVERSE CALCULATION & QC REPORT
=====================================================================
1. PROJECT METADATA & PARAMETERS
---------------------------------------------------------------------
Number of Stations/Legs   : {n}
Starting Benchmark (X, Y) : E = {start_x:.3f} m, N = {start_y:.3f} m
Initial Start Azimuth     : {start_azim:.4f}°
Selected Adjustment Rule  : {selected_method}
Target Specification      : 1 : {min_spec:,}

2. SECTION A: ANGULAR DISCREPANCY ANALYSIS
---------------------------------------------------------------------
Formula for Polygon Sum   : (N - 2) * 180° = ({n} - 2) * 180° = {theoretical_sum:.3f}°
Sum of Measured Angles    : {sum_measured_angles:.4f}°
Angular Discrepancy Error : {angular_error_deg:+.4f}° ({angular_error_sec:+.2f} arc-seconds)
Correction per Station    : {correction_per_angle:+.4f}° (Distributed equally across {n} stations)

3. SECTION B: MISCLOSURE & PRECISION ANALYSIS
---------------------------------------------------------------------
Total Perimeter           : {total_perimeter:.3f} m
Sum of Raw Latitudes      : {sum_lat:+.4f} m
Sum of Raw Departures     : {sum_dep:+.4f} m
Linear Misclosure Error   : {linear_misclosure:.4f} m
Achieved Precision Ratio  : 1 : {relative_precision:,.0f}
Quality Control Status    : [{qc_status}]

4. SECTION C: ADJUSTED STATION COORDINATES SHEET
---------------------------------------------------------------------
"""
        for i, row in df.iterrows():
            report_text += f"Line {row['Line']:<5} | Dist: {row['Distance']:6.2f}m | Azim: {row['Azimuth_Deg']:7.3f}° | Adj Lat: {row['Adj_Lat']:+8.3f} | Adj Dep: {row['Adj_Dep']:+8.3f} | Stn {station_letters[i]} E: {row['Easting_X']:10.3f} N: {row['Northing_Y']:10.3f}\n"

        report_text += f"Final Closing Stn {station_letters[-1]} | E: {eastings[-1]:10.3f} N: {northings[-1]:10.3f}\n"
        report_text += "=====================================================================\n"

        # Display Report in Colab Output
        display(Markdown(f"```text\n{report_text}\n```"))

        # Save files for download
        report_filename = "traverse_calculation_report.txt"
        with open(report_filename, "w") as f:
            f.write(report_text)

        output_csv = "traverse_coordinates_output.csv"
        df[['Line', 'Distance', 'Adj_Angle', 'Azimuth_Deg', 'Adj_Lat', 'Adj_Dep', 'Easting_X', 'Northing_Y']].round(3).to_csv(output_csv, index=False)

        # --- PLOTTING TRAVERSE LOOP ---
        plt.figure(figsize=(7, 7))
        plt.plot(eastings, northings, marker='o', color='darkblue', linestyle='-', linewidth=2, label='Adjusted Loop')

        station_labels = [f"Stn {station_letters[i]}" for i in range(n + 1)]
        for i, txt in enumerate(station_labels):
            plt.annotate(txt, (eastings[i], northings[i]), textcoords="offset points", xytext=(0,10), ha='center', fontweight='bold', color='firebrick')

        plt.title(f"Closed Traverse Network ({n} Stations) - {selected_method}")
        plt.xlabel("Easting (X) [m]")
        plt.ylabel("Northing (Y) [m]")
        plt.grid(True, linestyle='--', alpha=0.6)
        plt.legend()
        plt.tight_layout()

        plot_filename = "traverse_boundary_plot.png"
        plt.savefig(plot_filename, dpi=300)
        plt.show()

        # Trigger downloads
        files.download(report_filename)
        files.download(output_csv)
        files.download(plot_filename)
        print(f"\n[Success] Calculation report (.txt), coordinate sheet (.csv), and map plot (.png) successfully downloaded.")

compute_btn.on_click(on_compute_clicked)

# --- UI LAYOUT ---
dashboard_ui = widgets.VBox([
    widgets.HTML("<h3>Advanced Surveying Dashboard & Automated Report Generator</h3>"),
    widgets.HBox([num_stations_widget, start_x_widget, start_y_widget]),
    widgets.HBox([start_azim_widget, method_widget, spec_widget]),
    widgets.HBox([widgets.HTML("<b>Upload Survey CSV: </b>"), upload_widget]),
    widgets.HTML("<b>Or Enter Manually:</b>"),
    form_container,
    compute_btn
])

display(dashboard_ui, output_area)


Output()

In [ ]:
%%writefile app.py
import streamlit as st
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import io

st.set_page_config(page_title="Advanced Surveying Dashboard", layout="wide")

st.title("Advanced Surveying Dashboard & Automated Report Generator")

# --- Sidebar / Setup Configuration ---
st.sidebar.header("1. Base Configuration")
num_legs = st.sidebar.number_input("Number of Legs", min_value=3, max_value=50, value=5, step=1)
start_x = st.sidebar.number_input("Start Easting (X) [m]", value=1000.0)
start_y = st.sidebar.number_input("Start Northing (Y) [m]", value=1000.0)
start_azim = st.sidebar.number_input("Start Azimuth (°)", value=42.5)

selected_method = st.sidebar.selectbox(
    "Adjustment Method",
    options=['Bowditch (Compass Rule)', 'Transit Rule']
)

spec_label = st.sidebar.selectbox(
    "Target Spec Quality",
    options=['Cadastral Standard (1:10,000)', 'Engineering Control (1:20,000)', 'High-Precision Geodetic (1:50,000)']
)
spec_map = {
    'Cadastral Standard (1:10,000)': 10000,
    'Engineering Control (1:20,000)': 20000,
    'High-Precision Geodetic (1:50,000)': 50000
}
min_spec = spec_map[spec_label]

# --- File Upload Management ---
st.sidebar.write("---")
st.sidebar.header("2. Upload Survey CSV")
uploaded_file = st.sidebar.file_uploader("Upload a CSV (columns: Distance, Angle)", type=["csv"])

# Initialize lists for inputs
distances_list = [125.40 if i == 0 else 150.0 for i in range(num_legs)]
angles_list = [108.0] * num_legs

if uploaded_file is not None:
    try:
        df_uploaded = pd.read_csv(uploaded_file)
        df_uploaded.columns = [c.strip().lower() for c in df_uploaded.columns]
        if 'distance' in df_uploaded.columns and 'angle' in df_uploaded.columns:
            num_legs = len(df_uploaded)
            distances_list = df_uploaded['distance'].tolist()
            angles_list = df_uploaded['angle'].tolist()
            st.sidebar.success(f"Successfully loaded {num_legs} stations!")
        else:
            st.sidebar.error("CSV must contain 'Distance' and 'Angle' columns.")
    except Exception as e:
        st.sidebar.error(f"Error reading file: {e}")

# --- Interactive Input Form ---
st.header("Traverse Boundary Input Parameters")
station_letters = [chr(65 + i) for i in range(num_legs + 1)]

# Create columns for entering inputs side-by-side
col_dist, col_angle = st.columns(2)

final_distances = []
final_angles = []

with col_dist:
    st.subheader("Leg Distances (meters)")
    for i in range(num_legs):
        from_stn = station_letters[i]
        to_stn = station_letters[i+1]
        val = st.number_input(f"{from_stn}-{to_stn} Distance:", value=float(distances_list[i]), key=f"dist_{i}")
        final_distances.append(val)

with col_angle:
    st.subheader("Station Angles (degrees)")
    for i in range(num_legs):
        from_stn = station_letters[i]
        val = st.number_input(f"Stn {from_stn} Angle (°):", value=float(angles_list[i]), key=f"angle_{i}")
        final_angles.append(val)

# --- Computation Trigger ---
if st.button("Compute Traverse & Generate Report", type="primary"):
    n = num_legs
    distances = np.array(final_distances)
    measured_angles = np.array(final_angles)
    lines = [f"{station_letters[i]}-{station_letters[i+1]}" for i in range(n)]

    # 1. Angular Correction
    theoretical_sum = (n - 2) * 180.0
    sum_measured_angles = measured_angles.sum()
    angular_error_deg = sum_measured_angles - theoretical_sum
    angular_error_sec = angular_error_deg * 3600.0
    correction_per_angle = -angular_error_deg / n
    adjusted_angles = measured_angles + correction_per_angle

    # 2. Azimuth Propagation
    azimuths = [start_azim]
    curr_azim = start_azim
    for i in range(n - 1):
        curr_azim = (curr_azim + 180.0 - adjusted_angles[i]) % 360.0
        azimuths.append(curr_azim)

    df = pd.DataFrame({
        'Line': lines,
        'Distance': distances,
        'Meas_Angle': measured_angles,
        'Adj_Angle': adjusted_angles,
        'Azimuth_Deg': azimuths
    })

    # 3. Latitudes & Departures
    azimuth_rad = np.radians(df['Azimuth_Deg'])
    df['Raw_Lat'] = distances * np.cos(azimuth_rad)
    df['Raw_Dep'] = distances * np.sin(azimuth_rad)

    total_perimeter = distances.sum()
    sum_lat = df['Raw_Lat'].sum()
    sum_dep = df['Raw_Dep'].sum()
    linear_misclosure = np.sqrt(sum_lat**2 + sum_dep**2)
    relative_precision = total_perimeter / linear_misclosure if linear_misclosure != 0 else float('inf')

    # 4. Error Adjustments
    if selected_method == 'Bowditch (Compass Rule)':
        df['Corr_Lat'] = -sum_lat * (distances / total_perimeter)
        df['Corr_Dep'] = -sum_dep * (distances / total_perimeter)
    else:
        sum_abs_lat = np.abs(df['Raw_Lat']).sum()
        sum_abs_dep = np.abs(df['Raw_Dep']).sum()
        df['Corr_Lat'] = -sum_lat * (np.abs(df['Raw_Lat']) / sum_abs_lat) if sum_abs_lat != 0 else 0
        df['Corr_Dep'] = -sum_dep * (np.abs(df['Raw_Dep']) / sum_abs_dep) if sum_abs_dep != 0 else 0

    df['Adj_Lat'] = df['Raw_Lat'] + df['Corr_Lat']
    df['Adj_Dep'] = df['Raw_Dep'] + df['Corr_Dep']

    # 5. Adjusted Coordinates Mapping
    eastings = [start_x]
    northings = [start_y]
    curr_x, curr_y = start_x, start_y
    for dep, lat in zip(df['Adj_Dep'], df['Adj_Lat']):
        curr_x += dep
        curr_y += lat
        eastings.append(curr_x)
        northings.append(curr_y)

    df['Easting_X'] = eastings[:-1]
    df['Northing_Y'] = northings[:-1]

    # QC Criteria Evaluation
    qc_status = "PASS" if relative_precision >= min_spec else "FAIL (Exceeds Tolerance)"

    # Generate Analysis Text Report
    report_text = f"""=====================================================================
          AUTOMATED CLOSED TRAVERSE CALCULATION & QC REPORT
=====================================================================
1. PROJECT METADATA & PARAMETERS
---------------------------------------------------------------------
Number of Stations/Legs   : {n}
Starting Benchmark (X, Y) : E = {start_x:.3f} m, N = {start_y:.3f} m
Initial Start Azimuth     : {start_azim:.4f}°
Selected Adjustment Rule  : {selected_method}
Target Specification      : 1 : {min_spec:,}

2. SECTION A: ANGULAR DISCREPANCY ANALYSIS
---------------------------------------------------------------------
Formula for Polygon Sum   : (N - 2) * 180° = ({n} - 2) * 180° = {theoretical_sum:.3f}°
Sum of Measured Angles    : {sum_measured_angles:.4f}°
Angular Discrepancy Error : {angular_error_deg:+.4f}° ({angular_error_sec:+.2f} arc-seconds)
Correction per Station    : {correction_per_angle:+.4f}° (Distributed equally across {n} stations)

3. SECTION B: MISCLOSURE & PRECISION ANALYSIS
---------------------------------------------------------------------
Total Perimeter           : {total_perimeter:.3f} m
Sum of Raw Latitudes      : {sum_lat:+.4f} m
Sum of Raw Departures     : {sum_dep:+.4f} m
Linear Misclosure Error   : {linear_misclosure:.4f} m
Achieved Precision Ratio  : 1 : {relative_precision:,.0f}
Quality Control Status    : [{qc_status}]

4. SECTION C: ADJUSTED STATION COORDINATES SHEET
---------------------------------------------------------------------
"""
    for i, row in df.iterrows():
        report_text += f"Line {row['Line']:<5} | Dist: {row['Distance']:6.2f}m | Azim: {row['Azimuth_Deg']:7.3f}° | Adj Lat: {row['Adj_Lat']:+8.3f} | Adj Dep: {row['Adj_Dep']:+8.3f} | Stn {station_letters[i]} E: {row['Easting_X']:10.3f} N: {row['Northing_Y']:10.3f}\n"
    report_text += f"Final Closing Stn {station_letters[-1]} | E: {eastings[-1]:10.3f} N: {northings[-1]:10.3f}\n"
    report_text += "=====================================================================\n"

    # Display Output Sections in Streamlit UI
    st.header("Calculation Results & Plots")
    col_rep, col_plt = st.columns([1, 1])

    with col_rep:
        st.subheader("Survey Report")
        st.text_area("QC Summary", value=report_text, height=450)

    with col_plt:
        st.subheader("Traverse Polygon Loop")
        fig, ax = plt.subplots(figsize=(7, 7))
        ax.plot(eastings, northings, marker='o', color='darkblue', linestyle='-', linewidth=2, label='Adjusted Loop')

        station_labels = [f"Stn {station_letters[i]}" for i in range(n + 1)]
        for i, txt in enumerate(station_labels):
            ax.annotate(txt, (eastings[i], northings[i]), textcoords="offset points", xytext=(0,10), ha='center', fontweight='bold', color='firebrick')

        ax.set_title(f"Closed Traverse - {selected_method}")
        ax.set_xlabel("Easting (X) [m]")
        ax.set_ylabel("Northing (Y) [m]")
        ax.grid(True, linestyle='--', alpha=0.6)
        ax.legend()
        st.pyplot(fig)

    st.write("---")
    st.subheader("Download Center")
    col_dl1, col_dl2, col_dl3 = st.columns(3)

    # 1. Download Text Report
    col_dl1.download_button(
        label="Download Report (.txt)",
        data=report_text,
        file_name="traverse_calculation_report.txt",
        mime="text/plain"
    )

    # 2. Download Clean CSV Data
    clean_csv = df[['Line', 'Distance', 'Adj_Angle', 'Azimuth_Deg', 'Adj_Lat', 'Adj_Dep', 'Easting_X', 'Northing_Y']].round(3).to_csv(index=False)
    col_dl2.download_button(
        label="Download Coordinates (.csv)",
        data=clean_csv,
        file_name="traverse_coordinates_output.csv",
        mime="text/csv"
    )

    # 3. Download Image Plot
    img_buffer = io.BytesIO()
    fig.savefig(img_buffer, format="png", dpi=300)
    img_buffer.seek(0)
    col_dl3.download_button(
        label="Download Boundary Plot (.png)",
        data=img_buffer,
        file_name="traverse_boundary_plot.png",
        mime="image/png"
    )

Writing app.py
